# 合并预处理后的非药物处理组织样本数据

In [5]:
import scanpy as sc
import os
import re

In [2]:
# 数据路径
data_dir = "/mnt/c/Users/Future/Nutstore/1/UC/scRNA-seq_data/03_qc_filtered"

In [3]:
# 获取所有 h5ad 文件，排除 pbmc 数据
files = sorted([
    f for f in os.listdir(data_dir)
    if f.endswith(".h5ad") and "pbmc" not in f
])

In [6]:
def extract_dataset_id(fname: str) -> str:
    # 精确匹配 GSE 或 SCP 编号（大小写均可）
    m = re.search(r'(GSE\d+|SCP\d+)', fname, flags=re.IGNORECASE)
    return m.group(1).upper() if m else os.path.splitext(fname)[0]

# 读取数据并添加来源信息
adatas = []
for f in files:
    ad = sc.read_h5ad(os.path.join(data_dir, f))
    ds_id = extract_dataset_id(f)  # 例如：GSE125527 或 SCP259
    ad.obs['GSE'] = ds_id
    adatas.append(ad)
    print(f"Loaded {f} -> GSE column set to {ds_id} with {ad.n_obs} cells")

Loaded 10_GSE296969_qc.h5ad -> GSE column set to GSE296969 with 98112 cells
Loaded 11_GSE235663_qc.h5ad -> GSE column set to GSE235663 with 11042 cells
Loaded 1_GSE182270_qc.h5ad -> GSE column set to GSE182270 with 26128 cells
Loaded 2_GSE214695_qc.h5ad -> GSE column set to GSE214695 with 8908 cells
Loaded 3_GSE150115_qc.h5ad -> GSE column set to GSE150115 with 7230 cells
Loaded 4_GSE116222_qc.h5ad -> GSE column set to GSE116222 with 11113 cells
Loaded 5_GSE134649_qc.h5ad -> GSE column set to GSE134649 with 10272 cells
Loaded 6_GSE125527_tissue_qc.h5ad -> GSE column set to GSE125527 with 43241 cells
Loaded 7_GSE231993_qc.h5ad -> GSE column set to GSE231993 with 36404 cells
Loaded 8_SCP259_qc.h5ad -> GSE column set to SCP259 with 113095 cells
Loaded 9_GSE114374_qc.h5ad -> GSE column set to GSE114374 with 8962 cells


In [7]:
# 对所有数据取公共基因交集
common_genes = set(adatas[0].var_names)
for ad in adatas[1:]:
    common_genes &= set(ad.var_names)
adatas = [ad[:, list(common_genes)].copy() for ad in adatas]

In [8]:
# 合并为一个 AnnData
adata = adatas[0].concatenate(adatas[1:], batch_key="batch", index_unique=None)
print(f"Merged AnnData shape: {adata.shape}")

/tmp/ipykernel_10357/3806267993.py:2: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = adatas[0].concatenate(adatas[1:], batch_key="batch", index_unique=None)


Merged AnnData shape: (374507, 8243)


/home/future/miniconda3/envs/omicverse/lib/python3.10/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(


In [9]:
adata

AnnData object with n_obs × n_vars = 374507 × 8243
    obs: 'n_genes', 'GSM', 'nUMIs', 'mito_perc', 'detected_genes', 'cell_complexity', 'passing_mt', 'passing_nUMIs', 'passing_ngenes', 'doublet_score', 'predicted_doublet', 'group', 'GSE', 'batch'
    var: 'mt', 'gene_ids-0', 'feature_types-0', 'n_cells-0', 'n_cells-1', 'n_cells-10', 'gene_ids-2', 'feature_types-2', 'n_cells-2', 'gene_ids-3', 'feature_types-3', 'n_cells-3', 'n_cells-4', 'n_cells-5', 'gene_ids-6', 'feature_types-6', 'n_cells-6', 'n_cells-7', 'gene_ids-8', 'feature_types-8', 'n_cells-8', 'n_cells-9', 'features-9'

In [10]:
adata.obs.head()

,n_genes,GSM,nUMIs,mito_perc,detected_genes,cell_complexity,passing_mt,passing_nUMIs,passing_ngenes,doublet_score,predicted_doublet,group,GSE,batch
GSM8980878_UC68_CELL6_N4,790,GSM8980878,1128.0,0.079787,790,0.700355,True,True,True,0.024741,False,UC,GSE296969,0
GSM8980878_UC68_CELL10_N2,1807,GSM8980878,3051.0,0.077352,1807,0.592265,True,True,True,0.025539,False,UC,GSE296969,0
GSM8980878_UC68_CELL14_N2,3154,GSM8980878,6273.0,0.039694,3154,0.502790,True,True,True,0.011227,False,UC,GSE296969,0
GSM8980878_UC68_CELL20_N5,2054,GSM8980878,3109.0,0.019620,2054,0.660663,True,True,True,0.068769,False,UC,GSE296969,0
GSM8980878_UC68_CELL21_N3,4763,GSM8980878,11899.0,0.027986,4763,0.400286,True,True,True,0.046935,False,UC,GSE296969,0


In [11]:
adata.obs['GSE'].value_counts()

GSE
SCP259       113095
GSE296969     98112
GSE125527     43241
GSE231993     36404
GSE182270     26128
GSE116222     11113
GSE235663     11042
GSE134649     10272
GSE114374      8962
GSE214695      8908
GSE150115      7230
Name: count, dtype: int64

In [12]:
# var仅保留常用列
keep_vars = ['mt']  # 如果你在质控时标记过线粒体基因

adata.var = adata.var[keep_vars].copy()

print(f"Cleaned var columns: {adata.var.columns.tolist()}")


Cleaned var columns: ['mt']


In [13]:
adata

AnnData object with n_obs × n_vars = 374507 × 8243
    obs: 'n_genes', 'GSM', 'nUMIs', 'mito_perc', 'detected_genes', 'cell_complexity', 'passing_mt', 'passing_nUMIs', 'passing_ngenes', 'doublet_score', 'predicted_doublet', 'group', 'GSE', 'batch'
    var: 'mt'

In [15]:
# 创建保存目录
save_dir = "/mnt/c/Users/Future/Nutstore/1/UC/scRNA-seq_data/04_dataset_merge"
os.makedirs(save_dir, exist_ok=True)

# 保存合并后的 AnnData
save_path = os.path.join(save_dir, "tissue_dataset.h5ad")
adata.write_h5ad(save_path, compression='gzip')

print(f"✅ 合并后的数据已保存到: {save_path}")

✅ 合并后的数据已保存到: /mnt/c/Users/Future/Nutstore/1/UC/scRNA-seq_data/04_dataset_merge/tissue_dataset.h5ad
